# 实验一 静态网页爬取

- 姓名：洪英杰
- 学号：112404150212
- 班级：软件工程1242班
- GitHub：shihen123

本 notebook 实现两个任务：

| 任务 | 内容 | 教材出处 |
| --- | --- | --- |
| 任务 1 | QQ 表情包图片下载（批量、可复用） | 3.4 案例 1 |
| 任务 2 | 手机号码归属地查询（响应源码存本地 html） | 3.5 案例 2 |

运行方式：`Restart Kernel → Run All`。执行前确认内核为 **Python (lab01)**，
且 `lab01/.phone.local` 中已写入被测手机号（该文件不入库）。

In [1]:
# 环境与路径初始化：兼容 JupyterLab 从 lab01 启动、PyCharm 从仓库根启动两种情形
import sys
from pathlib import Path

BASE = Path.cwd()
if not (BASE / "spider_utils.py").exists():
    for parent in [BASE, *BASE.parents]:
        if (parent / "lab01" / "spider_utils.py").exists():
            BASE = parent / "lab01"
            break
sys.path.insert(0, str(BASE))

import re
import time
from urllib.parse import urlparse, urljoin

import config
from spider_utils import (
    FetchError, ensure_dir, fetch, get_image, mask_phone,
    now_iso, rel, save_image, save_json, save_text, sanitize, setup_logger,
)

log = setup_logger()
ensure_dir(config.EMOJI_DIR)

print("Python   :", sys.executable)
print("BASE_DIR :", config.BASE_DIR)
print("被测号码 :", mask_phone(config.target_phone()))

Python   : D:\PyCharmproject\project01\.venv\Scripts\python.exe
BASE_DIR : D:\PyCharmproject\project01\bigdata-collection-labs-112404150212\lab01
被测号码 : 193****4881


## 任务一　QQ 表情包图片下载

**目标页**：`http://www.bspider.top/yh31/`（课程镜像站，静态 HTML 列表页）

**思路**：
1. 请求列表页，得到整页 HTML；
2. 用正则切出每个 `<img>` 标签，取其 `src`（图片地址）与 `alt`（表情名称）；
3. 列表页的 `src` 是相对路径（`/static/yh31/images/xxx.gif`），必须用 `urljoin` 补全为绝对地址；
4. 逐张请求图片，用 `r.content` 取二进制，按 `序号_名称.扩展名` 存到 `output/qq_emojis/`。

下面这段是任务一的**完整代码**（解析 + 下载两段合在一起，便于截图）。

In [2]:
# ==================== 图片链接解析 ====================
IMG_TAG_RE = re.compile(r"<img\b[^>]*>", re.I)                        # 切出整个 <img ...> 标签
SRC_RE = re.compile(r'src="([^"]+\.(?:gif|png|jpg|jpeg))"', re.I)      # 取 src（限定图片扩展名）
ALT_RE = re.compile(r'alt="([^"]*)"', re.I)                            # 取 alt 作为表情名


def extract_images(html, base_url):
    """从列表页 HTML 中提取图片：返回 [{'name':..., 'url':...}, ...]，保序去重。"""
    items, seen = [], set()
    for tag in IMG_TAG_RE.findall(html):
        m = SRC_RE.search(tag)
        if not m:
            continue
        url = urljoin(base_url, m.group(1))          # 相对路径 -> 绝对地址
        if url in seen:
            continue
        seen.add(url)
        a = ALT_RE.search(tag)
        items.append({"name": a.group(1).strip() if a else "", "url": url})
    return items


# ==================== 批量下载（三段式） ====================
# ===== ① 定义请求信息 =====
def download_emojis(list_url, save_dir, limit=20):
    """批量下载表情包图片，返回下载清单。

    list_url : 列表页地址
    save_dir : 图片保存目录
    limit    : 最多下载多少张
    """
    headers = {**config.HEADERS_TASK1, "Referer": list_url}
    log.info("任务1 开始：%s", list_url)

    # ===== ② 发送请求并获得响应 =====
    resp = fetch(list_url, headers=headers, encoding="apparent")   # 列表页 HTML
    items = extract_images(resp.text, list_url)[:limit]            # 解析出图片地址
    log.info("列表页 %d 字符，命中图片 %d 条", len(resp.text), len(items))

    records = []
    for seq, item in enumerate(items, start=1):
        url, name = item["url"], item["name"] or "emoji_%d" % seq
        try:
            content, size = get_image(url, headers=headers)        # 图片二进制
            # ===== ③ 将响应图片存储到本地 =====
            suffix = Path(urlparse(url).path).suffix or ".gif"
            path = save_image(save_dir / ("%03d_%s%s" % (seq, sanitize(name), suffix)), content)
            records.append({
                "seq": seq, "name": name, "url": url, "file": rel(path),
                "ext": suffix.lstrip("."), "size_bytes": size, "status": "ok", "error": None,
            })
            log.info("[%02d] %s (%d 字节)", seq, path.name, size)
        except (FetchError, OSError) as err:                       # 单条失败不中断
            records.append({
                "seq": seq, "name": name, "url": url, "file": None, "ext": None,
                "size_bytes": 0, "status": "failed", "error": str(err),
            })
            log.warning("[%02d] 失败：%s", seq, err)
        time.sleep(config.POLITE_DELAY)                            # 限速
    return records


print("extract_images / download_emojis 已定义")

extract_images / download_emojis 已定义


In [3]:
records = download_emojis(config.LIST_URL, config.EMOJI_DIR, limit=config.DOWNLOAD_LIMIT)

ok = sum(1 for r in records if r["status"] == "ok")
print("\n共 %d 条，成功 %d 条，失败 %d 条\n" % (len(records), ok, len(records) - ok))

print("%-4s %-32s %-10s %s" % ("序号", "表情名称", "字节数", "本地文件"))
print("-" * 82)
for r in records[:8]:
    print("%-5s %-32s %-10s %s" % (r["seq"], r["name"][:30], r["size_bytes"], r["file"]))
if len(records) > 8:
    print("... 其余 %d 条略" % (len(records) - 8))

2026-11-06 17:29:52 [INFO] 任务1 开始：http://www.bspider.top/yh31/


2026-11-06 17:29:58 [INFO] 列表页 26086 字符，命中图片 20 条


2026-11-06 17:30:09 [INFO] [01] 001_小牛萌三的日常表情包.gif (94696 字节)


2026-11-06 17:30:15 [INFO] [02] 002_有关小兔吉祥的表情图片包.jpg (17612 字节)


2026-11-06 17:30:19 [INFO] [03] 003_有趣的大嘴蛙微信表情图片.gif (47792 字节)


2026-11-06 17:30:22 [INFO] [04] 004_快回家写作业.gif (53059 字节)


... (13 outputs left out)

2026-11-06 17:36:16 [INFO] [18] 018_天气好冷的图片，冷得发抖的可爱图片.gif (54038 字节)


2026-11-06 17:36:20 [INFO] [19] 019_告辞，后会有期.gif (35021 字节)


2026-11-06 17:36:23 [INFO] [20] 020_非常可爱的考拉小肉君qq表情包.gif (63099 字节)



共 20 条，成功 20 条，失败 0 条

序号   表情名称                             字节数        本地文件
----------------------------------------------------------------------------------
1     小牛萌三的日常表情包                       94696      output/qq_emojis/001_小牛萌三的日常表情包.gif
2     有关小兔吉祥的表情图片包                     17612      output/qq_emojis/002_有关小兔吉祥的表情图片包.jpg
3     有趣的大嘴蛙微信表情图片                     47792      output/qq_emojis/003_有趣的大嘴蛙微信表情图片.gif
4     快回家写作业                           53059      output/qq_emojis/004_快回家写作业.gif
5     知识就是力量                           56985      output/qq_emojis/005_知识就是力量.gif
6     握个手，很高兴认识你                       76500      output/qq_emojis/006_握个手，很高兴认识你.gif
7     你别走，再陪我玩一会                       114903     output/qq_emojis/007_你别走，再陪我玩一会.gif
8     奥利给                              83626      output/qq_emojis/008_奥利给.gif
... 其余 12 条略


In [4]:
devices = {
    "meta": {
        "task": "qq_emoji_download",
        "source": config.LIST_URL,
        "crawled_at": now_iso(),
        "count": len(records),
        "success": sum(1 for r in records if r["status"] == "ok"),
    },
    "items": records,
}
path = save_json(devices, config.DATA_DIR / "devices.json")
print("已写入：", rel(path))

files = sorted(config.EMOJI_DIR.iterdir())
total = sum(f.stat().st_size for f in files)
print("output/qq_emojis 下文件 %d 个，合计 %.1f KB" % (len(files), total / 1024))

已写入： data/devices.json
output/qq_emojis 下文件 20 个，合计 1499.6 KB


## 任务二　手机号码归属地查询

**目标页**：`https://www.ip138.com/mobile.asp`（教材 3.5.2 给出的实际查询地址）

> 注意：课程镜像站 `http://www.bspider.top/ip138/` 只是查询**表单页**的静态副本，
> 带参数请求它返回的仍是表单，拿不到结果，因此这里直接请求真实查询地址。

**思路**：
1. 用 `params={"mobile": 号码, "action": "mobile"}` 构造 GET 请求；
2. 把响应的**源代码**存成本地 `output/mobile_query.html`（实验硬性要求）；
3. 用「表格行」通用正则把结果表解析成字典，再映射到字段名。

结果表的每一行都是 `<tr><td>标签</td><td>值</td></tr>`，标签中带全角空格（如「运 营 商」），
比对前需去掉空格；「卡类型」对部分号码为空，属正常情况。

In [5]:
# ==================== 结果页解析 ====================
ROW_RE = re.compile(r"<tr>\s*<td>(.*?)</td>\s*<td>(.*?)</td>\s*</tr>", re.S)
TAG_RE = re.compile(r"<[^>]+>")

# 表格标签（去空格后） -> 记录字段名
FIELD_MAP = {
    "卡号归属地": "region",
    "运营商": "carrier",
    "卡类型": "card_type",
    "区号": "area_code",
    "邮编": "post_code",
}


def _clean(cell):
    """去掉单元格内的 HTML 标签、全角空格与首尾空白。"""
    return TAG_RE.sub("", cell).replace("\u3000", " ").strip()


def parse_mobile(html):
    """把归属地结果表解析成字典。"""
    rows = {}
    for label, value in ROW_RE.findall(html):
        rows[_clean(label).replace(" ", "")] = _clean(value)
    rec = {field: rows.get(label, "") for label, field in FIELD_MAP.items()}
    rec["status"] = "ok" if (rec["region"] or rec["carrier"]) else "parse_failed"
    return rec


# ==================== 查询（三段式） ====================
# ===== ① 定义请求信息 =====
def query_mobile(phone, html_path):
    """查询手机号归属地，把响应源码存成本地 html，并返回结构化记录。"""
    params = {"mobile": phone, "action": "mobile"}      # 教材 3.5.3 的两个查询参数
    headers = dict(config.HEADERS_TASK2)
    log.info("任务2 开始：查询 %s****%s", phone[:3], phone[-4:])

    # ===== ② 发送请求并获得响应 =====
    resp = fetch(config.IP138_URL, params=params, headers=headers, encoding="apparent")

    # ===== ③ 将响应源码存储到本地 html =====
    save_text(resp.text, html_path)
    log.info("响应 %d 字符，已存 %s", len(resp.text), rel(html_path))

    rec = parse_mobile(resp.text)
    rec["phone_masked"] = mask_phone(phone)
    rec["html_file"] = rel(html_path)
    return rec


print("parse_mobile / query_mobile 已定义")

parse_mobile / query_mobile 已定义


In [6]:
record = query_mobile(config.target_phone(), config.OUTPUT_DIR / "mobile_query.html")

print("\n查询结果（号码已脱敏）：")
for key, label in [("phone_masked", "手机号码"), ("region", "归属地"), ("carrier", "运营商"),
                   ("card_type", "卡类型"), ("area_code", "区号"), ("post_code", "邮编"),
                   ("status", "解析状态")]:
    value = record.get(key, "")
    print("  %-8s : %s" % (label, value if value != "" else "（空）"))

print("  源码文件 :", record["html_file"])

2026-11-06 17:36:23 [INFO] 任务2 开始：查询 193****4881


2026-11-06 17:36:23 [INFO] 响应 7477 字符，已存 output/mobile_query.html



查询结果（号码已脱敏）：
  手机号码     : 193****4881
  归属地      : 江苏 徐州
  运营商      : 中国电信
  卡类型      : （空）
  区号       : 0516
  邮编       : 221000
  解析状态     : ok
  源码文件 : output/mobile_query.html


In [7]:
companies = {
    "meta": {
        "task": "mobile_attribution",
        "source": config.IP138_URL,
        "queried_at": now_iso(),
    },
    "records": [record],
}
path = save_json(companies, config.DATA_DIR / "companies.json")
print("已写入：", rel(path))

已写入： data/companies.json


## 产物自检

In [8]:
import json


def _check_devices():
    data = json.loads((config.DATA_DIR / "devices.json").read_text(encoding="utf-8"))
    return data["meta"]["count"] == len(data["items"]) and data["meta"]["success"] >= 20


def _check_companies():
    data = json.loads((config.DATA_DIR / "companies.json").read_text(encoding="utf-8"))
    rec = data["records"][0]
    return rec["status"] == "ok" and bool(rec["region"]) and bool(rec["carrier"])


def _check_html():
    html = (config.OUTPUT_DIR / "mobile_query.html").read_text(encoding="utf-8")
    return "查询结果" in html and "卡号归属地" in html


checks = [
    ("图片目录文件数 >= 20", lambda: len(list(config.EMOJI_DIR.glob("*"))) >= 20),
    ("devices.json 可解析且计数一致", _check_devices),
    ("companies.json 可解析且 status=ok", _check_companies),
    ("mobile_query.html 含查询结果表格", _check_html),
]

print("%-34s %s" % ("检查项", "结果"))
print("-" * 46)
for name, fn in checks:
    try:
        passed = bool(fn())
    except Exception as err:
        passed = False
        print("  异常：", err)
    print("%-34s %s" % (name, "PASS" if passed else "FAIL"))

检查项                                结果
----------------------------------------------
图片目录文件数 >= 20                      PASS
devices.json 可解析且计数一致              PASS
companies.json 可解析且 status=ok      PASS
mobile_query.html 含查询结果表格          PASS
